# Chapitre 2. Probabilités conditionnelles, indépendance, Bayes

Calculer avec les probabilités totales et Bayes, et vérifier en simulant des clients.

*Cours « Probabilités pour la gestion des risques », M1 GRAF. Exécutez les cellules dans l'ordre (Maj + Entrée).*

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, integrate
from math import comb, perm, factorial, log, exp

rng = np.random.default_rng(42)   # graine : memes resultats a chaque execution

## Le test médical (question B4)

In [2]:
pM, sens, faux_pos = 0.01, 0.95, 0.05
pT = pM * sens + (1 - pM) * faux_pos
print('P(T) =', pT, '| P(M | T) =', pM * sens / pT)

P(T) = 0.059000000000000004 | P(M | T) = 0.16101694915254236


## Exemple résolu : Le raisonnement du bonus-malus

In [3]:
p_R, p_S_R, p_S_nonR = 0.3, 0.20, 0.05
p_S = p_R * p_S_R + (1 - p_R) * p_S_nonR
print('Exact : P(S) =', p_S, '| P(R | S) =', p_R * p_S_R / p_S)

n = 1_000_000
risque = rng.random(n) < p_R
sinistre = rng.random(n) < np.where(risque, p_S_R, p_S_nonR)
print('Simulation : P(S) =', sinistre.mean(), '| P(R | S) =', risque[sinistre].mean())

Exact : P(S) = 0.095 | P(R | S) = 0.631578947368421
Simulation : P(S) = 0.094981 | P(R | S) = 0.6301049683620935


## Indépendance : au moins un sinistre parmi n contrats

In [4]:
p = 0.05
for n_c in [1, 5, 10, 20, 50]:
    print(n_c, 'contrats :', round(1 - (1 - p)**n_c, 4))

1 contrats : 0.05
5 contrats : 0.2262
10 contrats : 0.4013
20 contrats : 0.6415
50 contrats : 0.9231


## Vérification des exercices 2.1 à 2.5

In [5]:
print('2.1 :', 0.5 * 0.4)
print('2.2 :', 0.2 * 0.6, 0.2 + 0.6 - 0.2 * 0.6, 0.2)

# 2.3 par simulation
d1 = rng.integers(1, 7, size=1_000_000); d2 = rng.integers(1, 7, size=1_000_000)
au_moins_un_6 = (d1 == 6) | (d2 == 6)
print('2.3 : exact', 1/11, '| simulation', np.mean((d1 == 6) & (d2 == 6)) / np.mean(au_moins_un_6))

parts = np.array([0.40, 0.35, 0.25]); fraude = np.array([0.02, 0.03, 0.05])
pf = np.sum(parts * fraude)
print('2.4 : P(fraude) =', pf, '| P(A3 | fraude) =', parts[2] * fraude[2] / pf)

pJ, sJ, sE = 0.2, 0.25, 0.08
pS1 = pJ * sJ + (1 - pJ) * sE
pS1S2 = pJ * sJ**2 + (1 - pJ) * sE**2
print('2.5 : (a)', pS1, '(b)', pJ * sJ / pS1, '(c)', pS1S2 / pS1)

# 2.5 par simulation : chaque assure garde sa classe deux ans
jeune = rng.random(n) < pJ
p_ind = np.where(jeune, sJ, sE)
S1 = rng.random(n) < p_ind; S2 = rng.random(n) < p_ind
print('2.5 simulation : P(S2 | S1) =', S2[S1].mean(), '| P(S2) =', S2.mean())

2.1 : 0.2
2.2 : 0.12 0.68 0.2
2.3 : exact 0.09090909090909091 | simulation 0.09071832017153915
2.4 : P(fraude) = 0.031 | P(A3 | fraude) = 0.40322580645161293
2.5 : (a) 0.114 (b) 0.4385964912280702 (c) 0.15456140350877193
2.5 simulation : P(S2 | S1) = 0.15381841211758704 | P(S2) = 0.113498
